# Prithvi Corn Yield — End-to-end run on SageMaker

**Recommended kernel:** PyTorch 2.4 GPU  
**Recommended instance:** `ml.g5.2xlarge` (24 GB A10G), **200+ GB EBS** (default 5 GB will fill instantly)

**Before running:**
1. Set `HF_TOKEN` as an env var on the instance — never paste it into a cell. From a SageMaker terminal:
   ```bash
   echo 'export HF_TOKEN=hf_xxx' >> ~/.bashrc && source ~/.bashrc
   ```
   Then restart the kernel so the variable is visible to Python.
2. Set `S3_URI` in Cell 3.
3. `S3_OUT_URI` in Cell 11 defaults to `${S3_URI}/outputs` — override if you want elsewhere.

Cells 1–4 are one-time setup. Cells 5–11 are the pipeline; safe to re-run from any cell forward.

## Cell 1 — Clone repo + install deps (one-time per instance)

In [ ]:
%%bash
cd ~
[ -d 2026-Hackathon-prompt-2 ] || git clone https://github.com/Alex0420W/2026-Hackathon-prompt-2.git 2026-Hackathon-prompt-2
cd 2026-Hackathon-prompt-2
pip install -q -r requirements.txt

## Cell 2 — cd into repo for the rest of the notebook

In [ ]:
import os
os.chdir(os.path.expanduser("~/2026-Hackathon-prompt-2"))
!pwd

## Cell 3 — Auto-discover + sync data from S3

Set `S3_URI` to the bucket+prefix you were given. The cell will:
1. List everything under that prefix
2. Detect which of the expected datasets are present
3. Sync each one in parallel to its local path
4. Print a summary table

`STATES_FILTER` only filters the giant `processed/chips` sync — useful for a smoke test on 1–2 states before committing 100+ GB.

In [ ]:
# === EDIT THESE TWO LINES ===
S3_URI = "s3://<bucket>/<prefix>"        # e.g. s3://csu-hackathon-2026/prompt-2/data
STATES_FILTER = None                     # None = all states; or e.g. ["IA", "NE"] for a smoke test
# ============================

WANT = {
    "raw/hls":       "data/raw/hls",
    "raw/cdl":       "data/raw/cdl",
    "raw/nass":      "data/raw/nass",       # needs yield_county.parquet AND yield_state.parquet
    "raw/gridmet":   "data/raw/gridmet",    # county_daily.parquet
    "raw/landsat":   "data/raw/landsat",    # county_month_vi.parquet
    "raw/usdm":      "data/raw/usdm",       # county_weekly.parquet
    "raw/gnatsgo":   "data/raw/gnatsgo",    # county_soil.parquet
    "processed/labels":   "data/processed/labels",
    "processed/features": "data/processed/features",
    "processed/chips":    "data/processed/chips",
}

import subprocess, concurrent.futures
from pathlib import Path

assert S3_URI.startswith("s3://") and "<bucket>" not in S3_URI, "set S3_URI first"
S3_URI = S3_URI.rstrip("/")

print(f"Listing {S3_URI}/ ...")
ls = subprocess.run(["aws", "s3", "ls", S3_URI + "/", "--recursive"],
                    capture_output=True, text=True, check=True)
all_keys = [line.split(None, 3)[-1] for line in ls.stdout.strip().splitlines() if line.strip()]
parts = S3_URI.replace("s3://", "").split("/", 1)
key_root = parts[1] + "/" if len(parts) > 1 else ""
rel_keys = [k[len(key_root):] for k in all_keys if k.startswith(key_root)]

present = {sub: any(k.startswith(sub + "/") for k in rel_keys) for sub in WANT}
print("\nFound in bucket:")
for sub, ok in present.items():
    print(f"  {'OK ' if ok else 'NO '}  {sub}")

for d in ("data/raw", "data/processed", "models", "reports"):
    Path(d).mkdir(parents=True, exist_ok=True)

jobs = []
for sub, local in WANT.items():
    if not present[sub]:
        continue
    cmd = ["aws", "s3", "sync", f"{S3_URI}/{sub}", local, "--no-progress"]
    if sub == "processed/chips" and STATES_FILTER:
        cmd += ["--exclude", "*"]
        for st in STATES_FILTER:
            cmd += ["--include", f"{st}/*"]
    jobs.append((sub, cmd))

def run(job):
    sub, cmd = job
    print(f"-> syncing {sub}")
    r = subprocess.run(cmd, capture_output=True, text=True)
    return sub, r.returncode, r.stderr[-500:] if r.returncode else "ok"

with concurrent.futures.ThreadPoolExecutor(max_workers=4) as ex:
    results = list(ex.map(run, jobs))

print("\nResults:")
for sub, rc, msg in results:
    print(f"  {'OK ' if rc == 0 else 'FAIL'}  {sub}  {msg if rc else ''}")

print("\nDisk usage:")
subprocess.run(["du", "-sh", "data/raw", "data/processed"])

## Cell 4 — Authenticate to Hugging Face

Reads `HF_TOKEN` from the environment. **Never paste the token into this cell.** Set it via terminal first (see notebook header).

In [ ]:
import os
from huggingface_hub import login

token = os.environ.get("HF_TOKEN")
assert token and token.startswith("hf_"), (
    "HF_TOKEN env var not set. Run in a terminal:\n"
    "  echo 'export HF_TOKEN=hf_xxx' >> ~/.bashrc && source ~/.bashrc\n"
    "then restart the kernel."
)
login(token=token, add_to_git_credential=False)
print("HF login OK")

## Cell 5 — Sanity check

Fail fast if paths or IAM aren't wired right. **Stop here if this errors.**

In [ ]:
!python notebooks/01_sanity_check.py

## Cell 6 — Build labels + features

In [ ]:
%%bash
PYTHONPATH=. python scripts/training/build_labels_metadata.py
PYTHONPATH=. python scripts/training/build_features.py
ls -lh data/processed/labels data/processed/features

## Cell 7 — Train Prithvi + LoRA

~4 hrs on `ml.g5.2xlarge`. Best checkpoint lands under `models/checkpoints/prithvi_yield/`. Logs tee'd to `reports/training_logs/run.log`.

In [ ]:
!mkdir -p reports/training_logs
!PYTHONPATH=. terratorch fit --config configs/terratorch_lora.yaml 2>&1 | tee reports/training_logs/run.log

## Cell 8 — Find best checkpoint by val_rmse

In [ ]:
from pathlib import Path
ckpts = list(Path("models/checkpoints/prithvi_yield").rglob("*.ckpt"))
ckpt = sorted(ckpts, key=lambda p: float(p.stem.split("rmse")[-1]))[0]
print("using:", ckpt)

## Cell 9 — Inference for 2025 + uncertainty cone

In [ ]:
!PYTHONPATH=. python scripts/inference/predict_2025.py --ckpt {ckpt}
!PYTHONPATH=. python scripts/training/analog_year_uncertainty.py

## Cell 10 — View results

In [ ]:
import pandas as pd
df = pd.read_parquet("reports/forecasts/yield_with_uncertainty_2025.parquet")
df.sort_values(["state", "checkpoint"])

## Cell 11 — Push outputs back to S3 (so they survive instance shutdown)

In [ ]:
S3_OUT_URI = f"{S3_URI}/outputs"   # or set explicitly e.g. "s3://my-bucket/team-alex/outputs"

import subprocess
subprocess.run(["aws", "s3", "sync", "reports", f"{S3_OUT_URI}/reports"], check=True)
subprocess.run(["aws", "s3", "sync", "models/checkpoints", f"{S3_OUT_URI}/models/checkpoints"], check=True)
print(f"Pushed to {S3_OUT_URI}")